# 01 - Exploratory Data Analysis (EDA)
===

Maritime Conflict Intelligence System (MCIS) へようこそ!

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.rcParams.update({
    'figure.dpi': 150,
    'figure.figsize': (10, 6),
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
})
sns.set_style('whitegrid')

DATA_DIR = Path('./data/processed')
OUTPUT_DIR = Path('./outputs/figures/eda')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
df = pd.read_parquet(DATA_DIR / 'ais_features.parquet')
print(f'Records: {len(df):,}')
print(f'Columns: {len(df.columns)}')
df.head(3)

In [ ]:
df.info()

In [ ]:
df.describe()

## Missing Values

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'missing': missing, 'pct': missing_pct})
missing_df = missing_df[missing_df['missing'] > 0].sort_values('pct', ascending=False)
missing_df

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=missing_df, y=missing_df.index, x='pct', ax=ax, palette='viridis')
ax.set_xlabel('Missing %')
ax.set_title('Missing Values by Column')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'missing_values.png', dpi=300)
plt.show()

## Vessel Types

In [ ]:
vessel_counts = df['VesselType'].value_counts().head(10)
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(x=vessel_counts.values, y=vessel_counts.index, ax=ax, palette='Blues_d')
ax.set_xlabel('Count')
ax.set_ylabel('Vessel Type')
ax.set_title('Top 10 Vessel Types')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'vessel_types.png', dpi=300)
plt.show()

## Speed Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(data=df, x='SOG', bins=50, ax=axes[0], color='teal', kde=True)
axes[0].set_xlabel('Speed Over Ground (knots)')
axes[0].set_title('Speed Distribution')

sns.boxplot(data=df, y='SOG', ax=axes[1], palette='viridis')
axes[1].set_ylabel('Speed (knots)')
axes[1].set_title('Speed Boxplot')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'speed_distribution.png', dpi=300)
plt.show()

## Geographic Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(
    df['LON'], df['LAT'],
    c=df['SOG'], cmap='viridis',
    alpha=0.6, s=20, edgecolor='white', linewidth=0.3
)
cbar = plt.colorbar(scatter)
cbar.set_label('Speed (knots)')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Vessel Traffic Distribution (Colored by Speed)')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'geographic_distribution.png', dpi=300)
plt.show()

## Temporal Distribution

In [ ]:
df['hour'] = df['BaseDateTime'].dt.hour
df['date'] = df['BaseDateTime'].dt.date

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

hourly = df.groupby('hour')['MMSI'].nunique()
sns.barplot(x=hourly.index, y=hourly.values, ax=axes[0], palette='coral')
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Unique Vessels')
axes[0].set_title('Hourly Vessel Activity')

daily = df.groupby('date')['MMSI'].nunique()
sns.lineplot(x=daily.index, y=daily.values, ax=axes[1], marker='o', color='teal')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Unique Vessels')
axes[1].set_title('Daily Vessel Activity')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'temporal_distribution.png', dpi=300)
plt.show()

## Conflict Zones

In [ ]:
zone_counts = df['conflict_zone_name'].value_counts()
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(x=zone_counts.values, y=zone_counts.index, ax=ax, palette='steelblue')
ax.set_xlabel('Records')
ax.set_ylabel('Conflict Zone')
ax.set_title('Records by Conflict Zone')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'conflict_zones.png', dpi=300)
plt.show()

## Correlation Matrix

In [ ]:
numeric_cols = ['SOG', 'COG', 'Heading', 'VesselType', 'Length', 'Width']
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax, 
            square=True, linewidths=0.5)
ax.set_title('Correlation Matrix')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'correlation_matrix.png', dpi=300)
plt.show()

In [ ]:
print('EDA Complete!')
print(f'Output: {OUTPUT_DIR}')